# Argus Panoptes — Day 3 DL: 1D-CNN / Spectrogram / Fusion + ONNX + Robustness

This notebook reproduces the **deep-learning** results end-to-end from the saved
artifacts (metrics JSONs, ONNX benchmark, robustness sweep), so it runs in
seconds without retraining. To regenerate the artifacts from scratch:

```bash
pip install -e ".[ml,dl]"
python scripts/generate_dataset.py --num-samples 2500 --output-dir data/dl_v1 \
    --seed 42 --duration-s 1.0 --extract-features --compute-spectrogram
python models/baseline.py  --data-dir data/dl_v1
python models/train_dl.py  --model 1dcnn       --data-dir data/dl_v1 --epochs 40
python models/train_dl.py  --model spectrogram --data-dir data/dl_v1 --epochs 40
python models/train_dl.py  --model fusion      --data-dir data/dl_v1 --epochs 40
python scripts/benchmark_onnx.py
python experiments/robustness_ablation.py --data-dir data/dl_v1
```

All three DL models share a multi-task head (regression: wear / cycle-time /
quality; classification: health-state) and are compared against the XGBoost
baseline **on the same held-out test split**.

In [1]:
import json
from pathlib import Path
import pandas as pd

EXP = Path.cwd().parents[0] if (Path.cwd().name == 'notebooks') else Path('experiments')
if not (EXP / 'dl_fusion_metrics.json').exists():
    EXP = Path('..')  # fallback when run from experiments/notebooks

def load(name):
    p = EXP / name
    return json.loads(p.read_text()) if p.exists() else None

dl = {m: load(f'dl_{m}_metrics.json') for m in ('1dcnn', 'spectrogram', 'fusion')}
bench = load('onnx_benchmark.json')
robust = load('robustness_results.json')
print('loaded:', {k: (v is not None) for k, v in dl.items()}, 'bench', bench is not None, 'robust', robust is not None)

loaded: {'1dcnn': True, 'spectrogram': True, 'fusion': True} bench True robust True


## 1. DL models vs XGBoost baseline (same test split)

In [2]:
rows = []
for name, d in dl.items():
    if d is None:
        continue
    w = d['dl_metrics']['regression']['wear_level']
    c = d['dl_metrics']['classification']
    rows.append({'model': name, 'params': d['n_params'], 'epochs': d['epochs_run'],
                 'wear_MAE': w['mae'], 'wear_R2': w['r2'],
                 'health_acc': c['accuracy'], 'health_F1': c['macro_f1']})
# XGBoost baseline (identical across DL runs; same split)
b = next((d['baseline_metrics'] for d in dl.values() if d and d.get('baseline_metrics')), None)
if b:
    rows.append({'model': 'xgboost', 'params': None, 'epochs': None,
                 'wear_MAE': b['regression']['wear_level']['mae'],
                 'wear_R2': b['regression']['wear_level']['r2'],
                 'health_acc': b['classification']['accuracy'],
                 'health_F1': b['classification']['macro_f1']})
pd.DataFrame(rows).set_index('model').round(4)

,params,epochs,wear_MAE,wear_R2,health_acc,health_F1
model,,,,,,
1dcnn,84807.0,14.0,0.2313,0.1064,0.314,0.2940
spectrogram,28135.0,29.0,0.2399,0.0765,0.300,0.2268
fusion,98855.0,30.0,0.1904,0.3516,0.402,0.3731
xgboost,NaN,NaN,0.0796,0.8863,0.658,0.6508


## 2. ONNX Runtime CPU latency (p50 / p95, chunk = 16384 samples)

In [3]:
if bench:
    lat = [{'model': m, 'single_p50_ms': r['single']['p50_ms'], 'single_p95_ms': r['single']['p95_ms'],
            'batch_p50_ms': r['batched']['p50_ms'], 'throughput_cps': r['batched']['throughput_cps']}
           for m, r in bench['models'].items()]
    if 'xgboost' in bench:
        x = bench['xgboost']
        lat.append({'model': 'xgboost(+DSP)', 'single_p50_ms': x['end_to_end_p50_ms'],
                    'single_p95_ms': None, 'batch_p50_ms': None, 'throughput_cps': None})
    display(pd.DataFrame(lat).set_index('model').round(3))

,single_p50_ms,single_p95_ms,batch_p50_ms,throughput_cps
model,,,,
1dcnn,0.371,0.481,14.599,2191.938
spectrogram,0.423,0.583,24.348,1314.260
fusion,0.381,0.561,15.063,2124.404
xgboost(+DSP),9.571,NaN,NaN,NaN


## 3. Noise-robustness ablation (XGBoost DSP vs 1D-CNN)

In [4]:
if robust:
    rdf = pd.DataFrame(robust['configs'])[['label', 'xgb_wear_mae', 'dl_wear_mae', 'xgb_health_f1', 'dl_health_f1']]
    display(rdf.set_index('label').round(4))
    print('Drift is rejected by the shared band-pass/detrend front-end; Gaussian noise')
    print('hurts the raw-waveform 1D-CNN far more than the integrated-band XGBoost path.')

,xgb_wear_mae,dl_wear_mae,xgb_health_f1,dl_health_f1
label,,,,
clean,0.0838,0.2366,0.6752,0.2714
gaussian sd=0.1*rms,0.0931,0.2493,0.6325,0.1761
gaussian sd=0.25*rms,0.1142,0.4445,0.5951,0.0842
gaussian sd=0.5*rms,0.1323,0.7423,0.5755,0.0775
drift 0.5*rms,0.0838,0.2366,0.6752,0.2714
drift 1.0*rms,0.0838,0.2366,0.6752,0.2714
quantize 8-bit,0.0843,0.2366,0.6675,0.2714
quantize 4-bit,0.0917,0.2351,0.6188,0.1994


Drift is rejected by the shared band-pass/detrend front-end; Gaussian noise
hurts the raw-waveform 1D-CNN far more than the integrated-band XGBoost path.
